# 01 — Ingest & Validate

Verify both parquets are built and match expected shape/date range.
Run the build scripts first if parquets are missing:
```
python src/processing/build_ontime_dmv.py
python src/processing/build_t100_dmv.py
```

In [1]:
import pandas as pd
from pathlib import Path

import sys
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from config import PROCESSED_DATA_PATH

PROC = PROJECT_ROOT / PROCESSED_DATA_PATH

In [2]:
ontime = pd.read_parquet(PROC / 'ontime_dmv.parquet')
ontime['FlightDate'] = pd.to_datetime(ontime['FlightDate'])
print('on-time:', ontime.shape)
print('  date range:', ontime['FlightDate'].min().date(), '->', ontime['FlightDate'].max().date())
print('  airports:', sorted(ontime['Origin'].unique()))
print()

on-time: (2823044, 33)
  date range: 2015-01-01 -> 2026-01-31
  airports: ['BWI', 'DCA', 'IAD']



In [3]:
t100 = pd.read_parquet(PROC / 't100_intl_dmv.parquet')
print('T-100 intl:', t100.shape)
print('  year range:', t100['YEAR'].min(), '->', t100['YEAR'].max())
print('  airports (ORIGIN):', sorted(t100[t100['ORIGIN'].isin(['IAD','DCA','BWI'])]['ORIGIN'].unique()))
print('  top dest countries:', t100['DEST_COUNTRY'].value_counts().head(8).to_dict())

T-100 intl: (35846, 23)
  year range: 2015 -> 2026
  airports (ORIGIN): ['BWI', 'DCA', 'IAD']
  top dest countries: {'US': 18268, 'CA': 2705, 'MX': 1376, 'GB': 1376, 'DE': 1053, 'FR': 600, 'DO': 579, 'SV': 543}


In [4]:
# Null audit — ontime delay cause columns
delay_cols = ['CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay']
print('on-time null rates:')
print(ontime[delay_cols].isnull().mean().round(3))

on-time null rates:
CarrierDelay         0.808
WeatherDelay         0.808
NASDelay             0.808
SecurityDelay        0.808
LateAircraftDelay    0.808
dtype: float64
